# Train (Kaggle)

Fine-tunes VideoMAE-Base or Video Swin-Tiny on ASL Citizen using Kaggle's free GPU quota.

**Why Kaggle rather than Colab for this:** the dataset is already attached read-only, so there is no download. On Colab the same run re-transfers ~83,000 videos every session, which on free tier can consume more time than it leaves for training. See D-009.

**Sessions end.** Kaggle caps a session at roughly 9-12 hours, and the free GPU quota is about 30 hours a week. A full baseline will not finish in one sitting, so this notebook is built to resume: it writes checkpoints to `/kaggle/working`, and picks up from a previous run's output when you attach it.

## Before running

1. **Add Data** → attach the ASL Citizen mirror.
2. **Accelerator** → GPU (P100 or T4). Changing this restarts the session and clears `/kaggle/working`.
3. **Internet** → On, so the repository can be cloned.
4. To continue a previous run, also attach that notebook's output under **Add Data → Your Work**.

## 1. Code

In [ ]:
# Environment guard. This notebook targets Kaggle; running it elsewhere
# fails later and less clearly than failing here.
import os

assert os.path.exists("/kaggle/input"), (
    "This is the KAGGLE notebook, but this runtime is not Kaggle.\n"
    "For Google Colab use notebooks/colab/02_train_colab.ipynb instead."
)
print("environment: kaggle")

REPO_URL = "https://github.com/Adgonzalez2018/ASL-Recognition-Model.git"

!git clone -q $REPO_URL /kaggle/working/asl
PROJECT = "/kaggle/working/asl/ASL_training"

!pip install -q -e $PROJECT --no-deps
!pip install -q av

In [ ]:
import torch

assert torch.cuda.is_available(), "No GPU. Settings -> Accelerator -> GPU."
print(torch.cuda.get_device_name(0))
print(f"{torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 2. Locate the dataset

In [ ]:
import os

DATASET_ROOT = None
for attachment in sorted(os.listdir("/kaggle/input")):
    for root, dirs, _files in os.walk(f"/kaggle/input/{attachment}"):
        if "splits" in dirs and "videos" in dirs:
            DATASET_ROOT = root
            break
    if DATASET_ROOT:
        break

assert DATASET_ROOT, "ASL Citizen not attached. Use Add Data in the sidebar."
print(f"dataset: {DATASET_ROOT}")

## 3. Manifests

Regenerated here rather than carried between sessions. `--probe-limit 0` skips video probing, which is the slow part, so this takes about a minute instead of the full audit's 30-60.

The result is byte-identical to what the full audit produced. The cell below proves it by comparing the generated identity against the one recorded in the committed audit report — if the dataset or the parsing has changed in any way, the hashes diverge and you find out here rather than after training.

In [ ]:
ARTIFACTS = "/kaggle/working/artifacts"

!cd $PROJECT && python scripts/audit_dataset.py \
    --dataset-root "$DATASET_ROOT" \
    --output-dir "$ARTIFACTS" \
    --write-manifests \
    --probe-limit 0 \
    --expected-classes 2731

In [ ]:
import json

# The committed audit is the reference: it probed every video and found the
# dataset clean. Matching identities means this session sees the same data.
with open(f"{PROJECT}/artifacts/audits/asl_citizen_audit.json") as handle:
    reference = json.load(handle)
with open(f"{ARTIFACTS}/audits/asl_citizen_audit.json") as handle:
    current = json.load(handle)

for name in ("label_map_identity", "manifest_identity"):
    match = reference[name] == current[name]
    print(f"{'OK  ' if match else 'DIFF'} {name}")
    print(f"       reference {reference[name]}")
    if not match:
        print(f"       this run  {current[name]}")

assert reference["manifest_identity"] == current["manifest_identity"], (
    "The manifests differ from the audited dataset. Do not train on this until "
    "the difference is understood."
)
assert not current["integrity"]["errors"], current["integrity"]["errors"]
print(f"\nintegrity clean, {current['counts']['manifest_records']:,} records")

## 4. Resume from a previous session

Kaggle clears `/kaggle/working` when a session ends. To continue a run, attach the previous notebook version's output under **Add Data → Your Work**; this cell copies its checkpoints back into place and training picks up from there.

On a first run it finds nothing and simply starts fresh.

In [ ]:
import shutil
from pathlib import Path

OUTPUTS = "/kaggle/working/outputs"
os.makedirs(OUTPUTS, exist_ok=True)

restored = []
for attachment in sorted(os.listdir("/kaggle/input")):
    previous = Path(f"/kaggle/input/{attachment}/outputs")
    if not previous.is_dir():
        continue
    for checkpoint in previous.rglob("latest.pt"):
        target = Path(OUTPUTS) / checkpoint.parent.relative_to(previous)
        target.mkdir(parents=True, exist_ok=True)
        for item in checkpoint.parent.iterdir():
            shutil.copy2(item, target / item.name)
        restored.append(str(target))

        run_dir = checkpoint.parent.parent
        for record in ("history.json", "run_metadata.json"):
            source = run_dir / record
            if source.exists():
                shutil.copy2(source, Path(OUTPUTS) / run_dir.relative_to(previous) / record)

if restored:
    print("Restored checkpoints; training will resume:")
    for path in restored:
        print(f"  {path}")
else:
    print("No previous checkpoints found. Starting a fresh run.")

## 5. Preflight

Measures what the run will cost before committing GPU quota to it. A few minutes.

Read three numbers:

1. **Epoch time** — multiply by your epoch count. If the total exceeds your weekly quota, reduce epochs or use the smaller architecture rather than discovering it at hour 25.
2. **Peak memory** — under 15% headroom warns. Lower `BATCH_SIZE` and raise `gradient_accumulation_steps` by the same factor so the effective batch stays comparable.
3. **Bottleneck** — decoding is CPU-bound, and Kaggle gives ~4 cores. If it says `data loading`, raising `--num-workers` helps more than any model change.

In [ ]:
# video_swin_tiny is ~30M parameters against VideoMAE's 88M, so it trains
# roughly 3x cheaper. A reasonable first baseline when GPU quota is limited.
MODEL = "videomae_base"  # or "video_swin_tiny"
BATCH_SIZE = 8
WORKERS = 4

!cd $PROJECT && python scripts/train_preflight.py \
    --model-config configs/models/$MODEL.yaml \
    --training-config configs/training/baseline.yaml \
    --artifacts-dir "$ARTIFACTS" \
    --dataset-root "$DATASET_ROOT" \
    --batch-size $BATCH_SIZE \
    --num-workers $WORKERS

## 6. Train

Set `EPOCHS` from what preflight measured, not from the config default.

Leave headroom: if a session is cut off mid-epoch you lose at most the work since the last checkpoint, which `checkpoint_every_minutes` bounds.

In [ ]:
EXPERIMENT = "exp-001-videomae-baseline"
RUN_NAME = "videomae-baseline-seed42"
EPOCHS = 20

!cd $PROJECT && python scripts/train.py \
    --model-config configs/models/$MODEL.yaml \
    --training-config configs/training/baseline.yaml \
    --artifacts-dir "$ARTIFACTS" \
    --dataset-root "$DATASET_ROOT" \
    --output-root "$OUTPUTS" \
    --experiment $EXPERIMENT \
    --run-name $RUN_NAME \
    --epochs $EPOCHS \
    --batch-size $BATCH_SIZE \
    --num-workers $WORKERS

## 7. Progress

In [ ]:
run_dir = f"{OUTPUTS}/{EXPERIMENT}/{RUN_NAME}"

with open(f"{run_dir}/history.json") as handle:
    history = json.load(handle)

for entry in history:
    line = f"epoch {entry['epoch']:3d}  loss {entry['train_loss']:.4f}"
    for name, value in entry["validation"].items():
        line += f"  {name} {value:.4f}"
    line += f"  ({entry['duration_seconds'] / 60:.0f} min)"
    if entry["non_finite_losses"]:
        line += f"  [{entry['non_finite_losses']} non-finite]"
    print(line)

with open(f"{run_dir}/run_metadata.json") as handle:
    metadata = json.load(handle)

print(f"\nrun kind        {metadata['run_kind']}")
print(f"effective batch {metadata['effective_batch_size']}")
print(f"precision       {metadata['precision_active']}")
print(f"gpu             {metadata['environment'].get('gpu')}")

!du -sh $run_dir/checkpoints 2>/dev/null

## 8. Save, so the next session can continue

**This is the step that makes resume work.** `/kaggle/working` is discarded when the session ends unless you save.

1. **Save Version → Save & Run All (Commit)**.
2. In the next session, **Add Data → Your Work → this notebook's output**.
3. Cell 4 finds the checkpoints and training continues from where it stopped.

The cell below removes the cloned repository from the output, since it is already on GitHub and only inflates the saved artifact.

One caution: *Save & Run All* re-executes every cell from the top. That is fine here — training resumes rather than restarting — but the session clock starts again.

## Notes

**Quota.** Kaggle grants roughly 30 GPU hours a week. Check remaining quota before starting a long run; being cut off mid-epoch is survivable, being cut off with no saved version is not.

**Out of memory.** Lower `BATCH_SIZE` and raise `gradient_accumulation_steps` in the training config by the same factor. Changing only the batch size makes the run non-comparable to others.

**The test split is untouched here.** Final test evaluation is a separate, deliberate step after model and threshold selection are fixed.

In [ ]:
# The repository is on GitHub; keeping a copy in the saved output wastes space.
!rm -rf /kaggle/working/asl
!du -sh /kaggle/working/* 2>/dev/null